# Connectome

Port of `packages/niivue/examples/connectome.html`. Loads a background volume and connectome mesh, with Jupyter controls for file choice, legend, shader, and connectome scaling.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"

nv = NiiVue(background_color=[0.15, 0.15, 0.25, 1.0], slice_type=4, backend="webgl2")

slice_type = widgets.Dropdown(options=[("Render", 4), ("A+C+S+R", 3)], value=4, description="Slice")
file_select = widgets.Dropdown(options=[("Dense", "connectome"), ("Sparse", "connectome2")], value="connectome", description="File")
legend = widgets.Checkbox(value=True, description="Legend")
shader = widgets.Dropdown(options=["phong", "toon", "outline"], value="phong", description="Shader")
node_scale = widgets.FloatSlider(value=3.0, min=0.5, max=10.0, step=0.5, description="Node")
edge_scale = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="Edge")

def load_connectome(_change=None):
    nv.load_meshes([
        {"url": f"{MESHES}/{file_select.value}.jcon", "isLegendVisible": legend.value}
    ])
    nv.set_mesh(0, {"shaderType": shader.value})
    nv.set_connectome_options(0, {"nodeScale": node_scale.value, "edgeScale": edge_scale.value})

def update_slice(change):
    nv.slice_type = change["new"]

def update_legend(change):
    nv.set_mesh(0, {"isLegendVisible": change["new"]})

def update_shader(change):
    nv.set_mesh(0, {"shaderType": change["new"]})

def update_node(change):
    nv.set_connectome_options(0, {"nodeScale": change["new"]})

def update_edge(change):
    nv.set_connectome_options(0, {"edgeScale": change["new"]})

slice_type.observe(update_slice, names="value")
file_select.observe(load_connectome, names="value")
legend.observe(update_legend, names="value")
shader.observe(update_shader, names="value")
node_scale.observe(update_node, names="value")
edge_scale.observe(update_edge, names="value")

controls = widgets.VBox([
    widgets.HBox([slice_type, file_select, legend, shader]),
    widgets.HBox([node_scale, edge_scale]),
])
display(controls)
display(nv)

nv.clip_plane_color = [0, 0, 0, 0]
nv.set_clip_planes([[0.15, 180, 0], [-0.1, 0, 0]])
nv.load_volumes([{"url": f"{VOLUMES}/mni152.nii.gz"}])
load_connectome()
